<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Assignment_4_Comparing_SMA_and_SES_Using_SMAPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Assignment: Comparing SMA and SES Using SMAPE
# Uploading and Loading the Dataset
from google.colab import files
uploaded = files.upload()

Saving monthly-milk-production-pounds.csv to monthly-milk-production-pounds (2).csv


In [8]:
# Importing libraries
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("monthly-milk-production-pounds.csv")

df.head()

,Month,Monthly milk production (pounds per cow)
0,1962-01,589
1,1962-02,561
2,1962-03,640
3,1962-04,656
4,1962-05,727


In [17]:
# Rename
df = df.rename(columns={
    'Monthly milk production (pounds per cow)': 'Production'
})

df.head()

,Month,Production
0,1962-01-01,589
1,1962-02-01,561
2,1962-03-01,640
3,1962-04-01,656
4,1962-05-01,727


In [18]:
# Parsing Dates and Sorting Data
# Converting 'Month' column to datetime
df['Month'] = pd.to_datetime(df['Month'])

# Sorting by Month
df = df.sort_values(by='Month')

# Resetting index
df = df.reset_index(drop=True)

df.head()

,Month,Production
0,1962-01-01,589
1,1962-02-01,561
2,1962-03-01,640
3,1962-04-01,656
4,1962-05-01,727


In [19]:
# Extracting First 20 Observations
df_subset = df.iloc[:20].copy()

df_subset

,Month,Production
0,1962-01-01,589
1,1962-02-01,561
2,1962-03-01,640
3,1962-04-01,656
4,1962-05-01,727
5,1962-06-01,697
6,1962-07-01,640
7,1962-08-01,599
8,1962-09-01,568
9,1962-10-01,577


In [20]:
# Simple Moving Average (SMA)
# Calculateing the 3-period SMA
df_subset['SMA'] = df_subset['Production'].rolling(window=3).mean()

# Shift to align forecast correctly (use past data only)
df_subset['SMA'] = df_subset['SMA'].shift(1)

df_subset

,Month,Production,SMA
0,1962-01-01,589,NaN
1,1962-02-01,561,NaN
2,1962-03-01,640,NaN
3,1962-04-01,656,596.666667
4,1962-05-01,727,619.000000
5,1962-06-01,697,674.333333
6,1962-07-01,640,693.333333
7,1962-08-01,599,688.000000
8,1962-09-01,568,645.333333
9,1962-10-01,577,602.333333


In [22]:
# Single Exponential Smoothing (SES)
# Setting smoothing factor
alpha = 0.3

# Creating empty SES column
df_subset['SES'] = np.nan

# Initializing the first forecast
df_subset.loc[0, 'SES'] = df_subset.loc[0, 'Production']

# Computing SES forecasts
for t in range(1, len(df_subset)):
    df_subset.loc[t, 'SES'] = (
        alpha * df_subset.loc[t-1, 'Production'] +
        (1 - alpha) * df_subset.loc[t-1, 'SES']
    )

df_subset

,Month,Production,SMA,SES
0,1962-01-01,589,NaN,589.000000
1,1962-02-01,561,NaN,589.000000
2,1962-03-01,640,NaN,580.600000
3,1962-04-01,656,596.666667,598.420000
4,1962-05-01,727,619.000000,615.694000
5,1962-06-01,697,674.333333,649.085800
6,1962-07-01,640,693.333333,663.460060
7,1962-08-01,599,688.000000,656.422042
8,1962-09-01,568,645.333333,639.195429
9,1962-10-01,577,602.333333,617.836801


In [23]:
# Defining SMAPE Function
def smape(actual, forecast):
    return 100/len(actual) * np.sum(
        2 * np.abs(forecast - actual) /
        (np.abs(actual) + np.abs(forecast))
    )

In [24]:
# Removing NaN Values Before Calculating SMAPE
# Dropping rows with NaN values
df_eval = df_subset.dropna()

df_eval

,Month,Production,SMA,SES
3,1962-04-01,656,596.666667,598.420000
4,1962-05-01,727,619.000000,615.694000
5,1962-06-01,697,674.333333,649.085800
6,1962-07-01,640,693.333333,663.460060
7,1962-08-01,599,688.000000,656.422042
8,1962-09-01,568,645.333333,639.195429
9,1962-10-01,577,602.333333,617.836801
10,1962-11-01,553,581.333333,605.585760
11,1962-12-01,582,566.000000,589.810032
12,1963-01-01,600,570.666667,587.467023


In [25]:
# Calculating SMAPE for Both Methods
sma_smape = smape(df_eval['Production'], df_eval['SMA'])
ses_smape = smape(df_eval['Production'], df_eval['SES'])

print("SMA SMAPE:", round(sma_smape, 2), "%")
print("SES SMAPE:", round(ses_smape, 2), "%")

SMA SMAPE: 8.55 %
SES SMAPE: 8.2 %
